In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

In [ ]:
CFG = {
    "seed": 42,
    "model_name": "distilbert-base-uncased",
    
    "train_dir": "/kaggle/input/datasets/xuntngtrn/mind-small-train-bert",
    "dev_dir": "/kaggle/input/datasets/xuntngtrn/mind-small-dev-bert",
    
    "max_len": 96,
    "batch_size": 16,
    "lr": 2e-5,
    "epochs": 10,
    "weight_decay": 0.01,
    
    "output_dir": "/kaggle/working/finetuned_distilbert_nrms",
    "use_title_only": False,   # True nếu muốn chỉ fine-tune bằng Title
}

set_seed(CFG["seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)

In [ ]:
def load_news(path_dir):
    cols = ['NewsID','Category','SubCategory','Title','Abstract','URL','TitleEntities','AbstractEntities']
    df = pd.read_csv(os.path.join(path_dir, 'news.tsv'), sep='\t', header=None)
    df.columns = cols
    df['Title'] = df['Title'].fillna("")
    df['Abstract'] = df['Abstract'].fillna("")
    return df

news_train = load_news(CFG["train_dir"])
news_dev = load_news(CFG["dev_dir"])

print(news_train.shape, news_dev.shape)
news_train.head()

In [ ]:
def build_text(df, use_title_only=False):
    if use_title_only:
        return df["Title"].astype(str)
    return (df["Title"].astype(str) + " [SEP] " + df["Abstract"].astype(str))

news_train = news_train.copy()
news_dev = news_dev.copy()

news_train["text"] = build_text(news_train, CFG["use_title_only"])
news_dev["text"] = build_text(news_dev, CFG["use_title_only"])

all_cats = sorted(set(news_train["Category"].astype(str)) | set(news_dev["Category"].astype(str)))
label2id = {c: i for i, c in enumerate(all_cats)}
id2label = {i: c for c, i in label2id.items()}

news_train["label"] = news_train["Category"].astype(str).map(label2id)
news_dev["label"] = news_dev["Category"].astype(str).map(label2id)

print("num_labels =", len(label2id))
print(label2id)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class NewsClsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
train_dataset = NewsClsDataset(
    news_train["text"],
    news_train["label"],
    tokenizer,
    CFG["max_len"]
)

eval_dataset = NewsClsDataset(
    news_dev["text"],
    news_dev["label"],
    tokenizer,
    CFG["max_len"]
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    per_device_train_batch_size=CFG["batch_size"],
    per_device_eval_batch_size=CFG["batch_size"],
    learning_rate=CFG["lr"],
    num_train_epochs=CFG["epochs"],
    weight_decay=CFG["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])

with open(os.path.join(CFG["output_dir"], "label2id.json"), "w") as f:
    json.dump(label2id, f, indent=2)

with open(os.path.join(CFG["output_dir"], "id2label.json"), "w") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)

with open(os.path.join(CFG["output_dir"], "ft_meta.json"), "w") as f:
    json.dump({
        "model_name": CFG["model_name"],
        "max_len": CFG["max_len"],
        "use_title_only": CFG["use_title_only"]
    }, f, indent=2)

print("Saved fine-tuned model to:", CFG["output_dir"])
print("Files:", os.listdir(CFG["output_dir"]))